In [1]:
"""
Build assumption_master sheet: profit_margins
-----
Extracts profit margins per product from the "Pricing structure" block
at rows 1-15 of the Assumptions sheet.

Schema: product | margin_type | margin | notes
margin_type ∈ {standard, dc}
"""

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME = "profit_margins"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")

Building: profit_margins
Source:   Basis and Assumptions.xlsx > Assumptions


In [2]:
def find_all_cells_any_col(ws, label, max_row=1200, max_col=30):
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits

def find_in_range(ws, label, row_start, row_end, col=None):
    for r, c in find_all_cells_any_col(ws, label):
        if row_start <= r < row_end and (col is None or c == col):
            return (r, c)
    return None

def get_col_letter(col):
    return openpyxl.utils.get_column_letter(col)

def peek(ws, row, col, span=4):
    """Show cells to the right of (row, col)."""
    parts = []
    for dc in range(0, span + 1):
        v = ws.cell(row=row, column=col + dc).value
        parts.append(f"{get_col_letter(col + dc)}{row}={v!r}")
    return " | ".join(parts)

# --- Diagnostic: locate the "Pricing structure" section anchors ---
print("=" * 90)
print("DIAGNOSTIC — Profit margin anchors")
print("=" * 90)

DIAGNOSTIC_LABELS = [
    "Profit Margin",
    "Product",
    "Profit Margin - DC",
    "Main Product indicater",   # source typo preserved
    "Stand-Alone LSDB Aids % of GLA",
    "Riders",
    "Main Product",
    "Main Fee PC",
    "Main Rand Amt",
    "GLA",  # will have MANY hits
]

for label in DIAGNOSTIC_LABELS:
    hits = find_all_cells_any_col(ws, label, max_row=50)  # only look at top of sheet
    print(f"\n'{label}': {len(hits)} match(es) in rows 1-50")
    for i, (r, c) in enumerate(hits[:6]):
        print(f"  [{i}] {peek(ws, r, c)}")

DIAGNOSTIC — Profit margin anchors

'Profit Margin': 1 match(es) in rows 1-50
  [0] C4='Profit Margin' | D4='Profit Margin - DC' | E4=None | F4='Main Product indicater ' | G4='Product'

'Product': 2 match(es) in rows 1-50
  [0] B4='Product' | C4='Profit Margin' | D4='Profit Margin - DC' | E4=None | F4='Main Product indicater '
  [1] G4='Product' | H4='% of NET Premium' | I4='Rand Fee Recurring' | J4='Stand-Alone LSDB Aids % of GLA' | K4=None

'Profit Margin - DC': 1 match(es) in rows 1-50
  [0] D4='Profit Margin - DC' | E4=None | F4='Main Product indicater ' | G4='Product' | H4='% of NET Premium'

'Main Product indicater': 1 match(es) in rows 1-50
  [0] F4='Main Product indicater ' | G4='Product' | H4='% of NET Premium' | I4='Rand Fee Recurring' | J4='Stand-Alone LSDB Aids % of GLA'

'Stand-Alone LSDB Aids % of GLA': 1 match(es) in rows 1-50
  [0] J4='Stand-Alone LSDB Aids % of GLA' | K4=None | L4='Riders' | M4='% Fee' | N4=None

'Riders': 1 match(es) in rows 1-50
  [0] L4='Riders' | M

In [3]:
# --- Mini-diagnostic: what's in column B rows 5-16? ---
print("Column B labels for Block 1 (Standard profit margins):")
print("-" * 60)
for row in range(5, 17):
    label = ws.cell(row=row, column=2).value
    profit_std = ws.cell(row=row, column=3).value
    profit_dc = ws.cell(row=row, column=4).value
    main_flag = ws.cell(row=row, column=6).value
    print(f"  B{row:2d}: label={label!r:35s} std={profit_std!r:10s} "
          f"dc={profit_dc!r:10s} main={main_flag!r}")

Column B labels for Block 1 (Standard profit margins):
------------------------------------------------------------
  B 5: label='GLA'                               std=0.06       dc=0.08       main=1
  B 6: label='FC'                                std=0.06       dc=None       main=None
  B 7: label='GIP'                               std=0.1        dc=None       main=None
  B 8: label='TIP'                               std=0.1        dc=None       main=None
  B 9: label='LSDB'                              std=0.075      dc=0.095      main=None
  B10: label='CI'                                std=0.075      dc=None       main=2
  B11: label='TI'                                std=0.075      dc=None       main=None
  B12: label='AccDth'                            std=0.06       dc=None       main=None
  B13: label='AccDB'                             std=0.075      dc=None       main=None
  B14: label='Dismemberment'                     std=0.075      dc=None       main=3
  B15: label=

In [4]:
# =============================================================================
# Product mapping: source label → canonical product key
# In-scope only. FC/GIP/TIP/SCIC/Ext FC are out of scope per handover.
# =============================================================================
PRODUCT_MAP = {
    "GLA":            "gla",
    "LSDB":           "lsdb",
    "CI":             "ci",
    "TI":             "ti",
    "AccDth":         "adb",
    "AccDB":          "accdb",
    "Dismemberment":  "dismem",
    # Explicitly out of scope:
    "FC":             None,
    "GIP":            None,
    "TIP":            None,
    "SCIC":           None,
    "Ext FC":         None,
}

# =============================================================================
# Harvest Block 1: rows 5-16, cols B (label), C (standard), D (DC)
# =============================================================================
BLOCK_ROW_START = 5
BLOCK_ROW_END   = 17  # exclusive

all_rows: list[dict] = []
skipped: list[tuple[str, str]] = []

for row in range(BLOCK_ROW_START, BLOCK_ROW_END):
    source_label = ws.cell(row=row, column=2).value  # col B
    if source_label is None:
        continue
    source_label = str(source_label).strip()

    if source_label not in PRODUCT_MAP:
        raise RuntimeError(
            f"Unexpected product label at B{row}: {source_label!r}. "
            f"Add to PRODUCT_MAP or mark as skip."
        )

    canonical = PRODUCT_MAP[source_label]
    if canonical is None:
        skipped.append((source_label, f"B{row}"))
        continue

    # Standard margin (col C)
    std_val = ws.cell(row=row, column=3).value
    if std_val is not None:
        all_rows.append({
            "product": canonical,
            "margin_type": "standard",
            "margin": std_val,
            "source_label": source_label,
            "source_row": row,
            "notes": "Standard product profit margin",
        })

    # DC margin (col D) — only emit row if source has a value
    dc_val = ws.cell(row=row, column=4).value
    if dc_val is not None:
        all_rows.append({
            "product": canonical,
            "margin_type": "dc",
            "margin": dc_val,
            "source_label": source_label,
            "source_row": row,
            "notes": "Defined Contribution variant profit margin",
        })

# =============================================================================
# Classify types (Python land, before polars upcast)
# =============================================================================
def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

for r in all_rows:
    v = r["margin"]
    r["param_type"] = classify(v)
    r["margin_str"] = "" if v is None else str(v)

df = pl.DataFrame(all_rows).select([
    "product",
    "margin_type",
    pl.col("margin_str").alias("margin"),
    "param_type",
    "source_label",
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
# Composite key must be unique
dupes = df.group_by(["product", "margin_type"]).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\n{dupes}"

# Expected products present
expected_products = {"gla", "lsdb", "ci", "ti", "adb", "accdb", "dismem"}
actual_products = set(df["product"].unique().to_list())
missing = expected_products - actual_products
unexpected = actual_products - expected_products
assert not missing, f"Missing products: {missing}"
assert not unexpected, f"Unexpected products: {unexpected}"

# All values numeric (no strings/nulls should have leaked through)
type_dist = df["param_type"].value_counts()
for row in type_dist.iter_rows(named=True):
    assert row["param_type"] in ("float", "int"), (
        f"Non-numeric margin found: type={row['param_type']!r}"
    )

# =============================================================================
# Frozen value expectations for A2026_07
# =============================================================================
def get_margin(product: str, margin_type: str) -> float:
    row = df.filter(
        (pl.col("product") == product) & (pl.col("margin_type") == margin_type)
    )
    assert row.height == 1, f"Missing: {product}/{margin_type}"
    return float(row["margin"][0])

frozen = [
    ("gla",    "standard", 0.06),
    ("gla",    "dc",       0.08),
    ("lsdb",   "standard", 0.075),
    ("lsdb",   "dc",       0.095),
    ("ci",     "standard", 0.075),
    ("ti",     "standard", 0.075),
    ("adb",    "standard", 0.06),
    ("accdb",  "standard", 0.075),
    ("dismem", "standard", 0.075),
]
for product, mtype, expected in frozen:
    actual = get_margin(product, mtype)
    assert abs(actual - expected) < 1e-9, (
        f"{product}/{mtype}: got {actual}, expected {expected}"
    )

# DC variant only exists for gla and lsdb — verify no other product has one
dc_products = set(df.filter(pl.col("margin_type") == "dc")["product"].to_list())
assert dc_products == {"gla", "lsdb"}, (
    f"Unexpected DC products: {dc_products - {'gla', 'lsdb'}}"
)

print(f"✓ {df.height} rows validated")
print(f"✓ {len(expected_products)} in-scope products present")
print(f"✓ {len(frozen)} frozen value checks passed")
print(f"✓ DC variant only for gla/lsdb (as expected)")
print(f"\nSkipped (out of scope): {[s[0] for s in skipped]}")
print(f"\nFinal DataFrame:")
print(df)

✓ 9 rows validated
✓ 7 in-scope products present
✓ 9 frozen value checks passed
✓ DC variant only for gla/lsdb (as expected)

Skipped (out of scope): ['FC', 'GIP', 'TIP', 'SCIC', 'Ext FC']

Final DataFrame:
shape: (9, 7)
┌─────────┬─────────────┬────────┬────────────┬───────────────┬────────────┬───────────────────────┐
│ product ┆ margin_type ┆ margin ┆ param_type ┆ source_label  ┆ source_row ┆ notes                 │
│ ---     ┆ ---         ┆ ---    ┆ ---        ┆ ---           ┆ ---        ┆ ---                   │
│ str     ┆ str         ┆ str    ┆ str        ┆ str           ┆ i64        ┆ str                   │
╞═════════╪═════════════╪════════╪════════════╪═══════════════╪════════════╪═══════════════════════╡
│ gla     ┆ standard    ┆ 0.06   ┆ float      ┆ GLA           ┆ 5          ┆ Standard product      │
│         ┆             ┆        ┆            ┆               ┆            ┆ profit margin         │
│ gla     ┆ dc          ┆ 0.08   ┆ float      ┆ GLA           ┆ 5       

In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote profit_margins.parquet
  Size:    2.90 KB
  SHA-256: f55eae3f01517f05...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
        "Standard and DC profit margins per product. Extracted from Pricing "
        "Structure block (rows 5-16, cols B-D). FC/GIP/TIP/SCIC/Ext FC "
        "explicitly skipped. DC variant only for gla/lsdb (source has blanks "
        "for others → no rows emitted per rule 5.5)."
    ),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated

BUILD COMPLETE: profit_margins
